In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

# Đảm bảo biểu đồ hiển thị ngay trong Notebook
%matplotlib inline

In [ ]:
def load_single_image(file_path):
    """Đọc một bức ảnh duy nhất từ đường dẫn cụ thể được cung cấp."""
    # Đọc ảnh dưới dạng grayscale
    img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
    
    if img is None:
        print(f"Lỗi: Không tìm thấy hoặc không thể đọc ảnh tại '{file_path}'. Vui lòng kiểm tra lại đường dẫn!")
        return None, None
        
    # Lấy tên file từ đường dẫn để hiển thị cho đẹp
    filename = os.path.basename(file_path)
    return filename, img

In [ ]:
def analyze_and_correct(img):
    """Đánh giá histogram và áp dụng phép biến đổi tương ứng."""
    mean_brightness = np.mean(img)
    std_dev = np.std(img)

    # Các ngưỡng phân loại
    DARK_THRESHOLD = 85
    BRIGHT_THRESHOLD = 170
    LOW_CONTRAST_THRESHOLD = 40

    if std_dev < LOW_CONTRAST_THRESHOLD:
        transformation_name = "Cân bằng Histogram (Tương phản thấp)"
        corrected_img = cv2.equalizeHist(img)
        
    elif mean_brightness < DARK_THRESHOLD:
        gamma = 0.5
        transformation_name = f"Gamma Correction (gamma={gamma}) (Quá tối)"
        table = np.array([((i / 255.0) ** gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
        corrected_img = cv2.LUT(img, table)
        
    elif mean_brightness > BRIGHT_THRESHOLD:
        gamma = 2.0
        transformation_name = f"Inverse Gamma (gamma={gamma}) (Quá sáng)"
        table = np.array([((i / 255.0) ** gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
        corrected_img = cv2.LUT(img, table)
        
    else:
        transformation_name = "Không can thiệp (Độ sáng/tương phản tốt)"
        corrected_img = img.copy()
        
    return corrected_img, transformation_name, mean_brightness, std_dev

In [ ]:
def visualize_results(filename, original_img, corrected_img, trans_name, mean, std):
    """Vẽ ảnh và histogram đối chiếu Trước/Sau."""
    plt.figure(figsize=(14, 8))
    plt.suptitle(f"Kết quả xử lý file: {filename}", fontsize=16, fontweight='bold')

    # 1. Ảnh Gốc
    plt.subplot(2, 2, 1)
    plt.imshow(original_img, cmap='gray', vmin=0, vmax=255)
    plt.title('Ảnh Gốc')
    plt.axis('off')

    # 2. Histogram Gốc
    plt.subplot(2, 2, 2)
    plt.hist(original_img.ravel(), bins=256, range=[0, 256], color='gray')
    plt.title(f'Histogram Gốc\nMean: {mean:.1f}, Std Dev: {std:.1f}')
    plt.xlabel('Cường độ Pixel')
    plt.ylabel('Tần suất')

    # 3. Ảnh Sau Xử Lý
    plt.subplot(2, 2, 3)
    plt.imshow(corrected_img, cmap='gray', vmin=0, vmax=255)
    plt.title(f'Ảnh Sau Xử Lý\n[{trans_name}]')
    plt.axis('off')

    # 4. Histogram Sau Xử Lý
    plt.subplot(2, 2, 4)
    plt.hist(corrected_img.ravel(), bins=256, range=[0, 256], color='gray')
    plt.title('Histogram Sau Xử Lý')
    plt.xlabel('Cường độ Pixel')
    plt.ylabel('Tần suất')

    plt.tight_layout()
    plt.show()

In [ ]:
# --- THIẾT LẬP ĐƯỜNG DẪN CỤ THỂ VÀ THƯ MỤC LƯU Ở ĐÂY ---
# Thay đổi đường dẫn tới file ảnh gốc của bạn
IMAGE_PATH = 'input_images/bai1/grayscale_image3.png'  # <-- Cập nhật đường dẫn này cho phù hợp với máy của bạn

# Thư mục để lưu các ảnh sau khi đã sửa
OUTPUT_FOLDER = 'output_images/bai1/grayscale_image3.png'  # <-- Cập nhật đường dẫn này cho phù hợp với máy của bạn

# Lệnh này kiểm tra xem thư mục lưu ảnh đã có chưa, nếu chưa có thì tự động tạo mới
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)
    print(f"Đã tạo thư mục mới: '{OUTPUT_FOLDER}'")

# 1. Đọc dữ liệu ảnh
filename, img = load_single_image(IMAGE_PATH)

# 2. Xử lý, Lưu và Hiển thị
if img is not None:
    print(f"Đã tải thành công ảnh '{filename}'. Đang tiến hành phân tích...\n")
    
    # Phân tích và sửa lỗi (gọi hàm từ Cell 3)
    corrected_img, trans_name, mean_val, std_val = analyze_and_correct(img)
    
    # --- PHẦN THÊM MỚI: LƯU ẢNH ---
    # Tạo tên file mới để phân biệt với ảnh gốc (VD: corrected_anh_thieu_sang_01.jpg)
    output_filename = f"corrected_{filename}"
    
    # Ghép tên thư mục và tên file thành một đường dẫn hoàn chỉnh an toàn
    output_path = os.path.join(OUTPUT_FOLDER, output_filename)
    
    # Lưu ảnh ra ổ cứng bằng cv2.imwrite
    success = cv2.imwrite(output_path, corrected_img)
    
    if success:
        print(f"✅ Thành công! Đã lưu ảnh sau khi sửa tại: {output_path}")
    else:
        print(f"❌ Lỗi: Không thể lưu ảnh. Vui lòng kiểm tra lại quyền ghi của thư mục!")
    # ------------------------------
    
    # Trực quan hóa kết quả lên Jupyter Notebook (gọi hàm từ Cell 4)
    visualize_results(filename, img, corrected_img, trans_name, mean_val, std_val)